In [15]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go


In [26]:
n_samples=1000
np.random.seed(42)
feature1=np.random.normal(0.05,1.2,n_samples)
feature2=feature1*0.85+np.random.normal(0,0.4,n_samples)

In [27]:
X=np.column_stack((feature1,feature2))
mu=np.mean(X,axis=0)
X_centered=X-mu

In [28]:
U,S,vt=np.linalg.svd(X_centered,full_matrices=False)

In [29]:
V=vt.T
eigenvalues=S**2/(n_samples-1)
eigenvalues

array([2.41446788, 0.09088551])

In [40]:
k=1
W=V[:,:k]

In [41]:
Z=X_centered @ W

In [42]:
X_reconstructed = Z @ W.T + mu

In [43]:
mse=np.mean((X-X_reconstructed)**2)
print("Mean Squared Error:", mse)

Mean Squared Error: 0.045397313203040315


In [44]:
fig = go.Figure()

    # A. Plot Original Data
fig.add_trace(go.Scatter(
    x=X[:, 0], y=X[:, 1],
    mode='markers',
    name='Original Data',
    marker=dict(color='royalblue', size=8, opacity=0.7),
    hovertemplate='Feature 1: %{x:.2f}<br>Feature 2: %{y:.2f}<extra></extra>'
))

# B. Plot Reconstructed Data (k=1)
fig.add_trace(go.Scatter(
    x=X_reconstructed[:, 0], y=X_reconstructed[:, 1],
    mode='markers',
    name='Reconstructed Data (k=1)',
    marker=dict(color='darkorange', symbol='x', size=6),
    hovertemplate='Recon F1: %{x:.2f}<br>Recon F2: %{y:.2f}<extra></extra>'
))

# C. Plot Reconstruction Error Lines (Orthogonal distances)
# We add the lines one by one, but group them in the legend
for i in range(n_samples):
    fig.add_trace(go.Scatter(
        x=[X[i, 0], X_reconstructed[i, 0]], 
        y=[X[i, 1], X_reconstructed [i, 1]],
        mode='lines',
        line=dict(color='gray', width=1, dash='dot'),
        showlegend=(i == 0), # Only show the first line in the legend
        name='Information Loss (Error)',
        legendgroup='error_lines',
        hoverinfo='skip'
    ))

# D. Plot Principal Components as Vectors (Arrows)
# Scale vectors by 2 standard deviations (2 * sqrt(eigenvalue)) for visibility
pc1_vec = V[:, 0] * np.sqrt(eigenvalues[0]) * 2
pc2_vec = V[:, 1] * np.sqrt(eigenvalues[1]) * 2

# PC1 Arrow
fig.add_annotation(
    x=mu[0] + pc1_vec[0], y=mu[1] + pc1_vec[1],
    ax=mu[0], ay=mu[1],
    xref='x', yref='y', axref='x', ayref='y',
    showarrow=True, arrowhead=2, arrowsize=1.5, arrowwidth=3, 
    arrowcolor='red', text='<b>PC1</b> (Max Variance)', font=dict(color='red', size=14)
)

# PC2 Arrow
fig.add_annotation(
    x=mu[0] + pc2_vec[0], y=mu[1] + pc2_vec[1],
    ax=mu[0], ay=mu[1],
    xref='x', yref='y', axref='x', ayref='y',
    showarrow=True, arrowhead=2, arrowsize=1.5, arrowwidth=3, 
    arrowcolor='green', text='<b>PC2</b> (Dropped)', font=dict(color='green', size=14)
)

# Plot the Mean center point
fig.add_trace(go.Scatter(
    x=[mu[0]], y=[mu[1]],
    mode='markers',
    name='Data Mean',
    marker=dict(color='black', symbol='star', size=12)
))

# Formatting the layout
fig.update_layout(
    title='Interactive PCA Visualization: Original vs Compressed Data',
    xaxis_title='Feature 1',
    yaxis_title='Feature 2',
    template='plotly_white',
    width=900,
    height=700,
    legend=dict(yanchor="top", y=0.99, xanchor="left", x=0.01, bgcolor="rgba(255, 255, 255, 0.8)")
)

# Force axes to have the same scale so orthogonal lines look perfectly perpendicular 90 degrees
fig.update_yaxes(scaleanchor="x", scaleratio=1)

# Open in browser
fig.show()